# 02 — Analyse de l'abstention

Analyse de l'abstention par département, corrélation avec la taille des communes,
et focus sur les non-votants de 18–39 ans.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_FILE = Path('../../public/cities/cities-data.json')

with open(DATA_FILE) as f:
    raw = json.load(f)

rows = []
for c in raw.values():
    t1 = c.get('Tour 1', {})
    t2 = c.get('Tour 2')
    analyse = c.get('Analyse', {})
    rows.append({
        'nom': c['nom_standard'],
        'code_dept': c['code_departement'],
        'dept': c['libelle_departement'],
        't1_inscrits': t1.get('Inscrits'),
        't1_pct_abs': t1.get('% Abs/Ins'),
        't2_pct_abs': t2.get('% Abs/Ins') if t2 else None,
        'non_votants_1839': analyse.get('Non votants de 18-39'),
        'pop_1839': analyse.get('Pop 18-39'),
        'pop_18plus': analyse.get('Pop 18+'),
        'part_ne_votant_pas': analyse.get('Part ne votant pas'),
    })

df = pd.DataFrame(rows)
df['pct_non_votants_1839'] = (df['non_votants_1839'] / df['pop_1839'] * 100).where(df['pop_1839'] > 0)
df['log_inscrits'] = np.log10(df['t1_inscrits'].clip(lower=1))
print(df.shape)

## Abstention moyenne par département

In [ ]:
dept_abs = (
    df.groupby(['code_dept', 'dept'])['t1_pct_abs']
    .agg(['mean', 'median', 'count'])
    .rename(columns={'mean': 'moy_abs', 'median': 'med_abs', 'count': 'nb_communes'})
    .reset_index()
    .sort_values('moy_abs', ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 16))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(dept_abs)))
bars = ax.barh(dept_abs['dept'], dept_abs['moy_abs'], color=colors, edgecolor='white')
ax.axvline(df['t1_pct_abs'].mean(), color='navy', linestyle='--', linewidth=1.5, label=f'Moyenne nationale ({df["t1_pct_abs"].mean():.1f}%)')
ax.set_xlabel("% Abstention moyen (Tour 1)")
ax.set_title("Abstention moyenne par département — Tour 1", fontsize=15)
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/02_abstention_dept.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 départements avec la plus forte abstention :")
print(dept_abs.head()[['dept', 'moy_abs', 'nb_communes']].to_string(index=False))
print("\nTop 5 départements avec la plus faible abstention :")
print(dept_abs.tail()[['dept', 'moy_abs', 'nb_communes']].to_string(index=False))

## Abstention vs taille de la commune

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    df['log_inscrits'], df['t1_pct_abs'],
    alpha=0.3, s=15, c=df['t1_pct_abs'], cmap='RdYlGn_r'
)
plt.colorbar(sc, ax=ax, label='% Abstention')

# Ligne de tendance
m = np.polyfit(df['log_inscrits'].dropna(), df['t1_pct_abs'].dropna(), 1)
x_line = np.linspace(df['log_inscrits'].min(), df['log_inscrits'].max(), 100)
ax.plot(x_line, np.polyval(m, x_line), 'navy', linewidth=2, label='Tendance')

ticks = [100, 500, 1000, 5000, 10000, 50000, 200000]
ax.set_xticks([np.log10(t) for t in ticks])
ax.set_xticklabels([f'{t:,}' for t in ticks], rotation=30)
ax.set_xlabel("Nombre d'inscrits (échelle log)")
ax.set_ylabel("% Abstention (Tour 1)")
ax.set_title("Abstention selon la taille de la commune")
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/02_abstention_taille.png', dpi=150, bbox_inches='tight')
plt.show()

corr = df[['log_inscrits', 't1_pct_abs']].corr().iloc[0, 1]
print(f"Corrélation taille/abstention : r = {corr:.3f}")

## Non-votants 18–39 ans

In [ ]:
df_jeunes = df.dropna(subset=['pct_non_votants_1839', 't1_pct_abs'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_jeunes['pct_non_votants_1839'], bins=50, color='tomato', edgecolor='white')
axes[0].set_title("% Non-votants 18–39 ans par commune")
axes[0].set_xlabel("% Non-votants 18–39")
axes[0].set_ylabel("Nb communes")

axes[1].scatter(
    df_jeunes['t1_pct_abs'], df_jeunes['pct_non_votants_1839'],
    alpha=0.3, s=12, color='tomato'
)
axes[1].set_xlabel("% Abstention globale (Tour 1)")
axes[1].set_ylabel("% Non-votants 18–39")
axes[1].set_title("Abstention globale vs Non-votants jeunes")

corr2 = df_jeunes[['t1_pct_abs', 'pct_non_votants_1839']].corr().iloc[0, 1]
axes[1].text(0.05, 0.92, f'r = {corr2:.3f}', transform=axes[1].transAxes, fontsize=12, color='navy')

plt.tight_layout()
plt.savefig('../outputs/02_non_votants_jeunes.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMoyenne nationale non-votants 18-39 : {df_jeunes['pct_non_votants_1839'].mean():.1f}%")
print(f"Médiane : {df_jeunes['pct_non_votants_1839'].median():.1f}%")

## Comparaison Tour 1 / Tour 2

In [ ]:
df_t2 = df.dropna(subset=['t1_pct_abs', 't2_pct_abs'])
df_t2 = df_t2[df_t2['t2_pct_abs'] > 0]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_t2['t1_pct_abs'], df_t2['t2_pct_abs'], alpha=0.3, s=12, color='steelblue')
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k--', linewidth=1, label='Tour 1 = Tour 2')
ax.set_xlabel("% Abstention Tour 1")
ax.set_ylabel("% Abstention Tour 2")
ax.set_title(f"Abstention Tour 1 vs Tour 2 ({len(df_t2):,} communes avec 2 tours)")
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/02_t1_vs_t2.png', dpi=150, bbox_inches='tight')
plt.show()

delta = df_t2['t2_pct_abs'] - df_t2['t1_pct_abs']
print(f"Communes avec hausse d'abstention au T2 : {(delta > 0).sum()} ({(delta > 0).mean()*100:.1f}%)")
print(f"Variation moyenne T2-T1 : {delta.mean():+.2f} points")